In [ ]:
# CELL 1 — Install Dependencies
!pip install einops yfinance

In [ ]:
# CELL 2 — Imports
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import yfinance as yf
from tqdm import tqdm

In [ ]:
# CELL 3 — Download 10‑Asset Yahoo Finance OHLCV Dataset
tickers = ["AAPL","MSFT","GOOG","AMZN","TSLA","NVDA","META","SPY","GLD","BTC-USD"]

df = yf.download(tickers, start="2010-01-01", auto_adjust=False)
df = df.ffill().dropna()  # forward-fill and drop any remaining NaNs

data = df.values.astype(np.float32)
print("Full dataset shape:", data.shape)  # e.g., (T, 60)

[*********************100%***********************]  10 of 10 completed

Full dataset shape: (4124, 60)


In [ ]:
# CELL 3.5 — Normalize the dataset (CRITICAL FIX)

mean = data.mean(axis=0, keepdims=True)
std = data.std(axis=0, keepdims=True) + 1e-6  # avoid divide-by-zero

data = (data - mean) / std

print("Data normalized. New mean (first 5):", data.mean(axis=0)[:5])
print("New std (first 5):", data.std(axis=0)[:5])

Data normalized. New mean (first 5): [-5.919996e-08  2.959998e-08  5.919996e-08 -1.479999e-07 -5.919996e-08]
New std (first 5): [1. 1. 1. 1. 1.]


In [ ]:
# CELL 4 — 80/10/10 Train/Val/Test Split
N = len(data)
train_end = int(N * 0.8)
val_end = int(N * 0.9)

train_data = data[:train_end]
val_data = data[train_end:val_end]
test_data = data[val_end:]

print("Train:", train_data.shape)
print("Val:  ", val_data.shape)
print("Test: ", test_data.shape)

Train: (3299, 60)
Val:   (412, 60)
Test:  (413, 60)


In [ ]:
# CELL 5 — Windowed Dataset Class
class WindowDataset(Dataset):
    def __init__(self, series, input_len=96, pred_len=24):
        self.series = series.astype(np.float32)
        self.input_len = input_len
        self.pred_len = pred_len

    def __len__(self):
        return len(self.series) - self.input_len - self.pred_len

    def __getitem__(self, idx):
        x = self.series[idx : idx + self.input_len]
        y = self.series[idx + self.input_len : idx + self.input_len + self.pred_len]
        return torch.tensor(x), torch.tensor(y)

In [ ]:
# CELL 6 — Create Datasets and Dataloaders
input_len = 96
pred_len = 24

train_ds = WindowDataset(train_data, input_len, pred_len)
val_ds = WindowDataset(val_data, input_len, pred_len)
test_ds = WindowDataset(test_data, input_len, pred_len)

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=32)
test_dl = DataLoader(test_ds, batch_size=32)

In [ ]:
# CELL 7 — FEDformer Model Definition (Multivariate-Correct)

class FourierBlock(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        # x: [B, L, D]
        x_ft = torch.fft.rfft(x, dim=1)
        x_ft = x_ft * 0.5
        x = torch.fft.irfft(x_ft, n=x.size(1), dim=1)
        return self.proj(x)


class FEDformer(nn.Module):
    def __init__(self, input_dim, d_model=64, depth=4, pred_len=24):
        super().__init__()
        self.input_dim = input_dim
        self.pred_len = pred_len

        # Embed 60 features → d_model
        self.embed = nn.Linear(input_dim, d_model)

        # Fourier layers
        self.layers = nn.ModuleList([FourierBlock(d_model) for _ in range(depth)])

        # Predict pred_len * input_dim = 24 * 60 = 1440 outputs
        self.head = nn.Linear(d_model, pred_len * input_dim)

    def forward(self, x):
        # x: [B, L, 60]
        x = self.embed(x)

        for layer in self.layers:
            x = x + layer(x)

        # Global pooling → [B, d_model]
        x = x.mean(dim=1)

        # Predict all future steps × all features
        x = self.head(x)  # [B, 1440]

        # Reshape → [B, 24, 60]
        return x.view(-1, self.pred_len, self.input_dim)

In [ ]:
# CELL 8 — Initialize Model, Optimizer, Loss
device = "cuda" if torch.cuda.is_available() else "cpu"

model = FEDformer(
    input_dim=train_data.shape[1],
    d_model=64,
    depth=4,
    pred_len=pred_len
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

def mae(pred, true):
    return torch.mean(torch.abs(pred - true))

In [ ]:
# CELL 9 — Training and Validation Functions
def train_epoch():
    model.train()
    total_loss = 0
    for x, y in tqdm(train_dl, desc="Training", leave=False):
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        pred = model(x)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(train_dl)


def validate_epoch():
    model.eval()
    mse_total = 0
    mae_total = 0
    with torch.no_grad():
        for x, y in tqdm(val_dl, desc="Validating", leave=False):
            x, y = x.to(device), y.to(device)
            pred = model(x)

            mse_total += criterion(pred, y).item()
            mae_total += mae(pred, y).item()

    return mse_total / len(val_dl), mae_total / len(val_dl)

In [ ]:
# CELL 10 — Run 10 Epochs (Multivariate-Compatible)

EPOCHS = 10

for epoch in range(1, EPOCHS + 1):
    print(f"\n===== Epoch {epoch}/{EPOCHS} =====")

    # Training pass
    train_loss = train_epoch()

    # Validation pass
    val_mse, val_mae = validate_epoch()

    # Epoch summary
    print(f"Train Loss: {train_loss:.6f}")
    print(f"Val MSE:    {val_mse:.6f}")
    print(f"Val MAE:    {val_mae:.6f}")


===== Epoch 1/10 =====


Train Loss: 0.245722
Val MSE:    0.427322
Val MAE:    0.496020

===== Epoch 2/10 =====


Train Loss: 0.159033
Val MSE:    0.397293
Val MAE:    0.485863

===== Epoch 3/10 =====


Train Loss: 0.149503
Val MSE:    0.428301
Val MAE:    0.510512

===== Epoch 4/10 =====


Train Loss: 0.145786
Val MSE:    0.365574
Val MAE:    0.472136

===== Epoch 5/10 =====


...

Train Loss: 0.141797
Val MSE:    0.376181
Val MAE:    0.467066

===== Epoch 7/10 =====


Train Loss: 0.141609
Val MSE:    0.438543
Val MAE:    0.498208

===== Epoch 8/10 =====


Train Loss: 0.140050
Val MSE:    0.459226
Val MAE:    0.499414

===== Epoch 9/10 =====


Train Loss: 0.139283
Val MSE:    0.438451
Val MAE:    0.491278

===== Epoch 10/10 =====


Train Loss: 0.138847
Val MSE:    0.459057
Val MAE:    0.483151


In [ ]:
# CELL 11 — Final Test Evaluation
def evaluate_test():
    model.eval()
    mse_total = 0
    mae_total = 0
    with torch.no_grad():
        for x, y in tqdm(test_dl, desc="Testing"):
            x, y = x.to(device), y.to(device)
            pred = model(x)

            mse_total += criterion(pred, y).item()
            mae_total += mae(pred, y).item()

    return mse_total / len(test_dl), mae_total / len(test_dl)

test_mse, test_mae = evaluate_test()
print("\n===== FINAL TEST RESULTS =====")
print(f"Test MSE: {test_mse:.6f}")
print(f"Test MAE: {test_mae:.6f}")

Testing: 100%|██████████| 10/10 [00:00<00:00, 353.99it/s]


===== FINAL TEST RESULTS =====
Test MSE: 1.858430
Test MAE: 1.122585
